**Start with cleaining audio files. All files are to be .WAV. All files are to be centered, cut to 3 seconds, and padded if too short.**

**YOU WILL SEE ERROR BUT IT IS WARNING FOR FUTURE DEPRECATION**

In [1]:
import os
import librosa
import soundfile as sf
import numpy as np

REAL_INPUT_DIR = "data/realAudio"
FAKE_INPUT_DIR = "data/fakeAudio"

CLEAN_REAL_DIR = "data/cleanReal"
CLEAN_FAKE_DIR = "data/cleanFake"

TARGET_SR = 16000
TARGET_DURATION = 3.0  # seconds


def standardize_audio(in_path, out_base, sr=TARGET_SR, duration=TARGET_DURATION):
    #Center-trim or pad audio to target duration and save as WAV
    try:
        y, _ = librosa.load(in_path, sr=sr, mono=True)
    except Exception as e:
        print(f"Could not load {in_path}: {e}")
        return

    target_len = int(sr * duration)

    if len(y) < target_len:
        # Center-pad
        pad_total = target_len - len(y)
        left = pad_total // 2
        right = pad_total - left
        y = np.pad(y, (left, right))
    else:
        # Take middle segment
        mid = len(y) // 2
        half = target_len // 2
        start = max(0, mid - half)
        end = start + target_len
        y = y[start:end]

    # Save as WAV
    out_path = out_base + ".wav"
    # Ensure directory exists
    out_dir = os.path.dirname(out_path)
    os.makedirs(out_dir, exist_ok=True)
    sf.write(out_path, y, sr)


def process_folder(input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    files = []
    for filename in os.listdir(input_dir):
        full_path = os.path.join(input_dir, filename)
        if os.path.isfile(full_path):
            files.append(full_path)
    if not files:
        print(f"No audio files found in {input_dir}")
        return
    for path in files:
        base_name = os.path.splitext(os.path.basename(path))[0]
        out_base = os.path.join(output_dir, base_name + "_clean")
        standardize_audio(path, out_base)
    print("All done!")


def main():
    print("Processing REAL audio…")
    process_folder(REAL_INPUT_DIR, CLEAN_REAL_DIR)

    print("\nProcessing FAKE audio…")
    process_folder(FAKE_INPUT_DIR, CLEAN_FAKE_DIR)


if __name__ == "__main__":
    main()

Processing REAL audio…


/var/folders/l1/991h4ps92ps2j8xfn3ycczzm0000gn/T/ipykernel_90267/3904874488.py:19: UserWarning: PySoundFile failed. Trying audioread instead.
  y, _ = librosa.load(in_path, sr=sr, mono=True)
/opt/anaconda3/lib/python3.13/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
/var/folders/l1/991h4ps92ps2j8xfn3ycczzm0000gn/T/ipykernel_90267/3904874488.py:19: UserWarning: PySoundFile failed. Trying audioread instead.
  y, _ = librosa.load(in_path, sr=sr, mono=True)
/opt/anaconda3/lib/python3.13/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Could not load data/realAudio/.DS_Store: 
All done!

Processing FAKE audio…
All done!


**Extract MFCC, Spectral, and Temporal features.**

In [2]:
import os
import numpy as np
import pandas as pd
import librosa

def extract_fingerprint(path, sr=16000, n_mfcc=13):
    y, sr = librosa.load(path, sr=sr, mono=True)
    features = {}

    # MFCC
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc, dct_type=2, norm="ortho")
    for i, row in enumerate(mfcc, start=1):
        features[f"mfcc_{i}_mean"] = float(row.mean())
        features[f"mfcc_{i}_std"]  = float(row.std())

    # Spectral Features
    # USE MEAN AND STD
    def add_stats(name, arr):
        features[f"{name}_mean"] = float(arr.mean())
        features[f"{name}_std"]  = float(arr.std())

    add_stats("spect_centroid", librosa.feature.spectral_centroid(y=y, sr=sr))
    add_stats("spect_bandwidth", librosa.feature.spectral_bandwidth(y=y, sr=sr))
    add_stats("spect_rolloff95", librosa.feature.spectral_rolloff(y=y, sr=sr, roll_percent=0.95))
    add_stats("spect_flatness", librosa.feature.spectral_flatness(y=y))

    #Temporal / Energy
    add_stats("zcr", librosa.feature.zero_crossing_rate(y))
    add_stats("rms", librosa.feature.rms(y=y))
    add_stats("spec_flux", librosa.onset.onset_strength(y=y, sr=sr))

    return features


def process_folder(dir_path, label):
    rows = []
    for filename in os.listdir(dir_path):
        if filename.lower().endswith(".wav"):
            filepath = os.path.join(dir_path, filename)
            features = extract_fingerprint(filepath)
            features["label"] = label
            features["file"] = filename
            rows.append(features)
    print(f"All done extracting features for {label}!")
    return rows


if __name__ == "__main__":
    REAL_DIR = "data/cleanReal"
    FAKE_DIR = "data/cleanFake"
    OUT = "data/spectralFingerprints.csv"

    rows = []
    rows += process_folder(REAL_DIR, "real")
    rows += process_folder(FAKE_DIR, "deepfake")
    
    df_original = pd.DataFrame(rows)
    df_original.to_csv(OUT, index=False)
    print(f"Saved to {OUT}")
    print("Extracted feature sample:\n", df_original.shape)
    print(df_original.head())

All done extracting features for real!
All done extracting features for deepfake!
Saved to data/spectralFingerprints.csv
Extracted feature sample:
 (60, 42)
   mfcc_1_mean  mfcc_1_std  mfcc_2_mean  mfcc_2_std  mfcc_3_mean  mfcc_3_std  \
0  -393.932587  130.114716    61.894531   58.727440    12.906281   40.271957   
1  -363.018616  193.994705    71.002953   66.365326   -12.329350   38.052834   
2  -553.867920  138.470261   117.140594   34.389332    11.266960   47.701668   
3  -489.287598  124.636398    78.815697   65.689674    -0.945584   45.948101   
4  -432.606354   59.131367   105.798805   39.436981    -3.878008   36.386127   

   mfcc_4_mean  mfcc_4_std  mfcc_5_mean  mfcc_5_std  ...  spect_flatness_mean  \
0    67.628304   48.064426     6.032644   20.635284  ...             0.015118   
1    18.733358   34.460884    -1.437612   23.611031  ...             0.018305   
2    21.686762   20.573704     2.251671   20.299982  ...             0.006886   
3    35.359287   44.800461    -9.83249

**This is the to determine the suitable number of PCA components.**

In [3]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import numpy as np

CSV_PATH = "data/spectralFingerprints.csv"

def find_optimal_pca(csv_path=CSV_PATH, variance_threshold=0.95):
    # Load data
    df = pd.read_csv(csv_path)

    # Separate features
    X = df.drop(columns=["label", "file"])

    # Standardize
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # PCA
    pca = PCA()
    pca.fit(X_scaled)

    # Explained variance
    variance = pca.explained_variance_ratio_
    cumulative = variance.cumsum()

    # Find smallest k such that cumulative variance >= threshold
    k = np.argmax(cumulative >= variance_threshold) + 1

    print(f"\nOptimal number of components for {variance_threshold} variance: {k}\n")

    return k


if __name__ == "__main__":
    k = find_optimal_pca()


Optimal number of components for 0.95 variance: 18



**This is the PCA algorithm.**

In [4]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import numpy as np

CSV_PATH = "data/spectralFingerprints.csv"

def run_pca(csv_path=CSV_PATH, n_components=k):
    # Load dataset
    df = pd.read_csv(csv_path)

    # Separate features and labels
    X = df.drop(columns=["label", "file"])
    y = df["label"]
    file_names = df["file"]

    feature_names = X.columns.tolist()

    # Standardize features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Apply PCA
    pca = PCA(n_components=n_components)
    X_pca = pca.fit_transform(X_scaled)

    # DISPLAY INFO
    print("\nExplained Variance Ratio")
    for i, var in enumerate(pca.explained_variance_ratio_):
        print(f"PC{i+1}: {var}")
    print("\nTOP CONTRIBUTING FEATURES PER PCA COMPONENT")
    loadings = pca.components_ 
    for i in range(n_components):
        component = loadings[i]
        # Sort features by absolute loading strength
        sorted_idx = np.argsort(-np.abs(component))
        top_idx = sorted_idx[:3]
        print(f"\nTop 3 features contributing to PC{i+1}:")
        for idx in top_idx:
            print(f"{feature_names[idx]} (loading={component[idx]})")

    # Build final PCA dataset
    df_pca = pd.DataFrame(X_pca, columns=[f"PC{i+1}" for i in range(n_components)])
    df_pca["label"] = y
    df_pca["file"] = file_names

    return df_pca


if __name__ == "__main__":
    df_pca = run_pca()
    print("\nPCA-Reduced DataFrame:")
    print(df_pca.head())


Explained Variance Ratio
PC1: 0.3236986117468391
PC2: 0.1299529529228754
PC3: 0.10818028018596601
PC4: 0.06603028205770366
PC5: 0.0581214319877544
PC6: 0.052385166227447606
PC7: 0.034816219421894196
PC8: 0.033584631600293686
PC9: 0.0244244046824415
PC10: 0.02222680912792974
PC11: 0.019161757985615407
PC12: 0.01809517296575658
PC13: 0.014993624371125261
PC14: 0.013742717610252404
PC15: 0.012945875565787288
PC16: 0.008943718690641023
PC17: 0.008510788269132276
PC18: 0.007358878386203376

TOP CONTRIBUTING FEATURES PER PCA COMPONENT

Top 3 features contributing to PC1:
spect_centroid_mean (loading=0.2556367707398309)
zcr_mean (loading=0.24625946697881299)
spect_rolloff95_mean (loading=0.2446366213988774)

Top 3 features contributing to PC2:
rms_mean (loading=0.3559041476672489)
mfcc_6_mean (loading=-0.3154371068636724)
rms_std (loading=0.3016505270056856)

Top 3 features contributing to PC3:
spect_rolloff95_std (loading=0.40387618003803566)
spect_bandwidth_std (loading=0.37722345634261295

**This is the OPTICS algorithm for min_samples=2 and eps=8 using cosine distance.**

In [12]:
from sklearn.cluster import OPTICS
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np

CSV_PATH = "data/spectralFingerprints.csv"

# Load data
df = pd.read_csv(CSV_PATH)

# Separate features and non-features
X = df.drop(columns=["label", "file"])
y = df["label"]

# SCALE FEATURES
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Run OPTICS on scaled data
optics = OPTICS(
    max_eps=8,
    min_samples=2,
    metric='cosine',
    min_cluster_size=2
)

optics.fit(X_scaled)

labels = optics.labels_
df["cluster"] = labels

# Check if any real clusters exist
non_noise_labels = [lab for lab in labels if lab != -1]

if len(non_noise_labels) == 0:
    print("\n[WARNING] OPTICS FOUND NO CLUSTERS — ALL POINTS LABELED -1")
    print("OPTICS is NOT removing outliers. Returning full dataset.")
    df_final = df.drop(columns=["cluster"])

else:
    # Identify outliers
    indices_to_drop = df[df["cluster"] == -1].index
    print("\nOUTLIER INDICES TO DROP (-1 labels):")
    print(indices_to_drop.tolist())

    print(f"\nOutliers detected: {len(indices_to_drop)}")
    print(f"Total samples: {len(df)}")
    print(f"Remaining samples: {len(df) - len(indices_to_drop)}")

    # Drop outliers
    df_new = df.drop(indices_to_drop).reset_index(drop=True)

    # Final cleaned dataset
    df_optics = df_new.drop(columns=["cluster"])


OUTLIER INDICES TO DROP (-1 labels):
[4, 6, 11, 12, 13, 15, 17, 19, 21, 22, 23, 25, 26, 29, 33, 39, 40, 42, 43, 50, 51, 54]

Outliers detected: 22
Total samples: 60
Remaining samples: 38


**This is the isolation forest that gives us the outliers.**

In [6]:
from sklearn.ensemble import IsolationForest
import pandas as pd
import numpy as np

CSV_PATH = "data/spectralFingerprints.csv"
df = pd.read_csv(CSV_PATH)

# Separate features/labels
X = df.drop(columns=["label", "file"])
y = df["label"]

clf = IsolationForest(random_state = 42 )

outlier_flags = clf.fit_predict(X)   # -1 = outlier, 1 = inlier
df['outlier'] = outlier_flags

# Summary counts\
n_outliers = np.sum(outlier_flags == -1)
n_inliers = np.sum(outlier_flags == 1)

print("OUTLIER STATISTICS")
print(f"Total outliers detected: {n_outliers}")
print(f"Total inliers: {n_inliers}")

# Show which indices will be dropped
indices_to_drop = df[df['outlier'] == -1].index
print("OUTLIER INDICES")
print(indices_to_drop.tolist())

# Drop outliers
df_new = df.drop(indices_to_drop)

# Final cleaned dataset
df_isolationforest = df_new.drop(columns=["outlier"])

OUTLIER STATISTICS
Total outliers detected: 11
Total inliers: 49
OUTLIER INDICES
[1, 2, 5, 8, 10, 13, 18, 20, 21, 25, 51]


**KNN model for k=5. Run on all modified data sets.**

In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split, LeaveOneOut, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Path to your CSV from the feature-extraction script
CSV_PATH = "data/spectralFingerprints.csv"


def run_loo(X, y, n_neighbors=5):
    #Perform Leave one out evaluation and return average accuracy
    loo = LeaveOneOut()
    num_splits = loo.get_n_splits(X)
    avg_result = 0
    for train_index, test_index in loo.split(X):
        # Split data
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]
        # Scale inside the fold
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        # Train model
        knn = KNeighborsClassifier(n_neighbors=n_neighbors)
        knn.fit(X_train_scaled, y_train)
        # Evaluate
        acc = knn.score(X_test_scaled, y_test)
        avg_result += acc
    return avg_result / num_splits

def run_knn(data, name):
    print(f"======= RUNNING KNN FOR DATA SET: {name} ========\n")
    # Load data
    X = data.drop(columns=["label", "file"])
    y = data["label"]
    
    # LOO Evaluation
    avg_loo_acc = run_loo(X, y)
    print("AVERAGE Accuracy with every datapoint as the test sample:")
    print(avg_loo_acc)

    # NORMAL RUN
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y)

    # Scale train and test
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # KNN classifier
    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(X_train_scaled, y_train)

    cv_scores = cross_val_score(knn, X_train_scaled, y_train, cv=5)
    print(f"\nCross validation scores:\n{cv_scores}\n")

    # Evaluate on held-out test set
    acc = knn.score(X_test_scaled, y_test)
    print(f"OVERALL Accuracy: {acc}\n")

    y_pred = knn.predict(X_test_scaled)

    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))


def main():
    run_knn(df_original, "ORIGINAL")
    run_knn(df_pca, "PCA")
    run_knn(df_optics, "OPTICS")
    run_knn(df_isolationforest, "ISOLATION FOREST")


if __name__ == "__main__":
    main()

======= RUNNING KNN FOR DATA SET: ORIGINAL ========

AVERAGE Accuracy with every datapoint as the test sample:
0.8666666666666667

Cross validation scores:
[0.77777778 0.88888889 1.         1.         0.625     ]

OVERALL Accuracy: 0.8888888888888888

Confusion Matrix:
[[9 0]
 [2 7]]

Classification Report:
              precision    recall  f1-score   support

    deepfake       0.82      1.00      0.90         9
        real       1.00      0.78      0.88         9

    accuracy                           0.89        18
   macro avg       0.91      0.89      0.89        18
weighted avg       0.91      0.89      0.89        18

======= RUNNING KNN FOR DATA SET: PCA ========

AVERAGE Accuracy with every datapoint as the test sample:
0.7166666666666667

Cross validation scores:
[0.55555556 0.88888889 0.625      0.625      0.75      ]

OVERALL Accuracy: 0.6111111111111112

Confusion Matrix:
[[8 1]
 [6 3]]

Classification Report:
              precision    recall  f1-score   support

    d

**RESULTS FOR MCKINLEY ONLY**

In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split, LeaveOneOut, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Path to your CSV from the feature-extraction script
CSV_PATH = "data/spectralFingerprints.csv"


def run_loo(X, y, n_neighbors=5):
    #Perform Leave one out evaluation and return average accuracy
    loo = LeaveOneOut()
    num_splits = loo.get_n_splits(X)
    avg_result = 0
    for train_index, test_index in loo.split(X):
        # Split data
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]
        # Scale inside the fold
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        # Train model
        knn = KNeighborsClassifier(n_neighbors=n_neighbors)
        knn.fit(X_train_scaled, y_train)
        # Evaluate
        acc = knn.score(X_test_scaled, y_test)
        avg_result += acc
    return avg_result / num_splits

def run_knn(data, name):
    print(f"======= RUNNING KNN FOR DATA SET: {name} ========\n")
    # Load data
    X = data.drop(columns=["label", "file", "speaker"])
    y = data["label"]
    
    # LOO Evaluation
    avg_loo_acc = run_loo(X, y)
    print("AVERAGE Accuracy with every datapoint as the test sample:")
    print(avg_loo_acc)

    # NORMAL RUN
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y)

    # Scale train and test
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # KNN classifier
    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(X_train_scaled, y_train)

    cv_scores = cross_val_score(knn, X_train_scaled, y_train, cv=5)
    print(f"\nCross validation scores:\n{cv_scores}\n")

    # Evaluate on held-out test set
    acc = knn.score(X_test_scaled, y_test)
    print(f"OVERALL Accuracy: {acc}\n")

    y_pred = knn.predict(X_test_scaled)

    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))


def main():
    df = pd.read_csv(CSV_PATH)
    # Derive speaker column from filename
    df["speaker"] = np.where('m' == df.file.str[0], 'Z', 'M')
    # Keep only speaker M
    df_M = df.loc[df.speaker == 'M']
    run_knn(df_M, "ORIGINAL DATA FOR SPEAKER: MCKINLEY")

if __name__ == "__main__":
    main()

======= RUNNING KNN FOR DATA SET: ORIGINAL DATA FOR SPEAKER: MCKINLEY ========

AVERAGE Accuracy with every datapoint as the test sample:
0.8

Cross validation scores:
[0.6  1.   0.75 0.75 1.  ]

OVERALL Accuracy: 0.6666666666666666

Confusion Matrix:
[[5 0]
 [3 1]]

Classification Report:
              precision    recall  f1-score   support

    deepfake       0.62      1.00      0.77         5
        real       1.00      0.25      0.40         4

    accuracy                           0.67         9
   macro avg       0.81      0.62      0.58         9
weighted avg       0.79      0.67      0.61         9



**RESULTS FOR ZACH ONLY**

In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split, LeaveOneOut, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Path to your CSV from the feature-extraction script
CSV_PATH = "data/spectralFingerprints.csv"


def run_loo(X, y, n_neighbors=5):
    #Perform Leave one out evaluation and return average accuracy
    loo = LeaveOneOut()
    num_splits = loo.get_n_splits(X)
    avg_result = 0
    for train_index, test_index in loo.split(X):
        # Split data
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]
        # Scale inside the fold
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        # Train model
        knn = KNeighborsClassifier(n_neighbors=n_neighbors)
        knn.fit(X_train_scaled, y_train)
        # Evaluate
        acc = knn.score(X_test_scaled, y_test)
        avg_result += acc
    return avg_result / num_splits

def run_knn(data, name):
    print(f"======= RUNNING KNN FOR DATA SET: {name} ========\n")
    # Load data
    X = data.drop(columns=["label", "file", "speaker"])
    y = data["label"]
    
    # LOO Evaluation
    avg_loo_acc = run_loo(X, y)
    print("AVERAGE Accuracy with every datapoint as the test sample:")
    print(avg_loo_acc)

    # NORMAL RUN
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y)

    # Scale train and test
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # KNN classifier
    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(X_train_scaled, y_train)

    cv_scores = cross_val_score(knn, X_train_scaled, y_train, cv=5)
    print(f"\nCross validation scores:\n{cv_scores}\n")

    # Evaluate on held-out test set
    acc = knn.score(X_test_scaled, y_test)
    print(f"OVERALL Accuracy: {acc}\n")

    y_pred = knn.predict(X_test_scaled)

    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))


def main():
    df = pd.read_csv(CSV_PATH)
    # Derive speaker column from filename
    df["speaker"] = np.where('m' == df.file.str[0], 'Z', 'M')
    # Keep only speaker Z
    df_Z = df.loc[df.speaker == 'Z']
    run_knn(df_Z, "ORIGINAL DATA FOR SPEAKER: ZACH")

if __name__ == "__main__":
    main()

======= RUNNING KNN FOR DATA SET: ORIGINAL DATA FOR SPEAKER: ZACH ========

AVERAGE Accuracy with every datapoint as the test sample:
1.0

Cross validation scores:
[1. 1. 1. 1. 1.]

OVERALL Accuracy: 0.8888888888888888

Confusion Matrix:
[[4 1]
 [0 4]]

Classification Report:
              precision    recall  f1-score   support

    deepfake       1.00      0.80      0.89         5
        real       0.80      1.00      0.89         4

    accuracy                           0.89         9
   macro avg       0.90      0.90      0.89         9
weighted avg       0.91      0.89      0.89         9

